In [ ]:
# Imports and GPU verification
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Load pre-trained ResNet-50 and modify final classifier layer
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)
model = model.to(device)

In [ ]:
# Load EuroSAT RGB dataset
dataset_dict = load_dataset("blanchon/EuroSAT_RGB")

if len(dataset_dict) == 1:
    full_data = dataset_dict[list(dataset_dict.keys())[0]]
else:
    full_data = concatenate_datasets([dataset_dict[split] for split in dataset_dict.keys()])

print(f"Total dataset size: {len(full_data)}")

In [ ]:
# Define image augmentations and PyTorch dataset wrapper
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class EuroSATDataset(Dataset):
    def __init__(self, hf_subset, transform=None):
        self.hf_subset = hf_subset
        self.transform = transform

    def __len__(self):
        return len(self.hf_subset)

    def __getitem__(self, idx):
        item = self.hf_subset[idx]
        image = item['image']
        label = item['label']
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
# Create reproducible 80/10/10 train, val, and test splits
total_size = len(full_data)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)

generator = torch.Generator().manual_seed(42)
indices = torch.randperm(total_size, generator=generator).tolist()

train_idx = indices[:train_size]
val_idx = indices[train_size:train_size + val_size]
test_idx = indices[train_size + val_size:]

train_loader = DataLoader(EuroSATDataset(full_data.select(train_idx), train_transform), batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(EuroSATDataset(full_data.select(val_idx), eval_transform), batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(EuroSATDataset(full_data.select(test_idx), eval_transform), batch_size=32, shuffle=False, num_workers=2)

In [ ]:
# Initial training loop on frozen model
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

history = {'train_loss': [], 'val_acc': []}

for epoch in range(5):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    avg_loss = running_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    print(f"Epoch [{epoch+1}/5] | Train Loss: {avg_loss:.4f} | Validation Accuracy: {val_acc*100:.2f}%")

In [ ]:
# Unfreeze all layers and fine-tune whole model at lower learning rate
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(5):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total
    avg_loss = running_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    print(f"Fine-tune Epoch [{epoch+1}/5] | Train Loss: {avg_loss:.4f} | Validation Accuracy: {val_acc*100:.2f}%")

In [ ]:
# Test evaluation and metrics calculation
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

class_names = ['AnnualCrop', 'Forest', 'HerbVeg', 'Highway', 'Industrial', 'Pasture', 'PermCrop', 'Residential', 'River', 'SeaLake']

print(f"Final Test Accuracy: {accuracy_score(all_labels, all_preds)*100:.2f}%\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

In [ ]:
# Generate and save research figures
epochs_range = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history['train_loss'], 'b-o', label='Train Loss')
plt.axvline(x=5.5, color='r', linestyle='--', label='Fine-tuning Starts')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, [acc * 100 for acc in history['val_acc']], 'g-o', label='Val Accuracy')
plt.axvline(x=5.5, color='r', linestyle='--', label='Fine-tuning Starts')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Validation Accuracy Curve')
plt.legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=300)
plt.show()

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('EuroSAT Confusion Matrix')
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

In [ ]:
# Grid cropping, temporal inference, and change mapping function
def process_and_compare_sat_images(img_path_t1, img_path_t2, patch_size=64):
    img_t1 = Image.open(img_path_t1).convert('RGB')
    img_t2 = Image.open(img_path_t2).convert('RGB')

    w, h = img_t1.size
    grid_x, grid_y = w // patch_size, h // patch_size

    map_t1 = np.zeros((grid_y, grid_x), dtype=int)
    map_t2 = np.zeros((grid_y, grid_x), dtype=int)
    deforestation_mask = np.zeros((grid_y, grid_x), dtype=int)

    model.eval()
    forest_idx = class_names.index('Forest')
    non_forest_indices = [class_names.index(c) for c in ['AnnualCrop', 'Industrial', 'Pasture', 'Residential', 'HerbVeg', 'PermCrop']]

    with torch.no_grad():
        for i in range(grid_y):
            for j in range(grid_x):
                box = (j * patch_size, i * patch_size, (j + 1) * patch_size, (i + 1) * patch_size)

                p1 = eval_transform(img_t1.crop(box)).unsqueeze(0).to(device)
                p2 = eval_transform(img_t2.crop(box)).unsqueeze(0).to(device)

                c1 = torch.max(model(p1), 1)[1].item()
                c2 = torch.max(model(p2), 1)[1].item()

                map_t1[i, j] = c1
                map_t2[i, j] = c2

                if c1 == forest_idx and c2 in non_forest_indices:
                    deforestation_mask[i, j] = 1

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(img_t1)
    axes[0].set_title('Year 1 Image (Baseline)')
    axes[1].imshow(img_t2)
    axes[1].set_title('Year 2 Image (Recent)')
    axes[2].imshow(deforestation_mask, cmap='Reds')
    axes[2].set_title('Model Detected Deforestation Mask')
    plt.savefig('deforestation_detection_result.png', dpi=300)
    plt.show()

    print(f"Total grid patches analyzed: {grid_x * grid_y}")
    print(f"Deforested patches detected: {np.sum(deforestation_mask)}")

process_and_compare_sat_images('/content/2018-10-05-France-Troyes.jpg', '/content/2026-10-06-France-Troyes.jpg')